In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [3]:
df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")

In [4]:
df.shape

(253680, 22)

In [5]:
df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_012          253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  

In [7]:
df["Diabetes_012"].value_counts()

Diabetes_012
0.0    213703
2.0     35346
1.0      4631
Name: count, dtype: int64

In [8]:
df["Diabetes_012"].value_counts(normalize=True)

Diabetes_012
0.0    0.842412
2.0    0.139333
1.0    0.018255
Name: proportion, dtype: float64

The dataset shows class imbalance across the three categories.

In [9]:
df.isnull().sum()

Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

In [10]:
df.duplicated().sum()

23899

In [11]:
df = df.drop_duplicates()

In [12]:
X = df.drop(columns=["Diabetes_012"])
y = df["Diabetes_012"]

In [13]:
sensitive_features = ["Sex", "Age", "Income"]

In [14]:
df["Sex"].value_counts()
df["Age"].value_counts().sort_index()
df["Income"].value_counts().sort_index()

Income
1.0     9792
2.0    11757
3.0    15922
4.0    19957
5.0    25345
6.0    35001
7.0    40189
8.0    71818
Name: count, dtype: int64

In [15]:
pd.crosstab(df["Sex"], df["Diabetes_012"], normalize="index")

Diabetes_012,0.0,1.0,2.0
Sex,,,
0.0,0.837421,0.020209,0.142370
1.0,0.813955,0.020064,0.165981


In [16]:
pd.crosstab(df["Age"], df["Diabetes_012"], normalize="index")

Diabetes_012,0.0,1.0,2.0
Age,,,
1.0,0.982039,0.003810,0.014151
2.0,0.972552,0.007640,0.019808
3.0,0.961496,0.007182,0.031322
4.0,0.937306,0.011607,0.051087
5.0,0.913737,0.011601,0.074662
6.0,0.881323,0.018036,0.100642
7.0,0.849179,0.018064,0.132757
8.0,0.824512,0.020146,0.155342
9.0,0.785344,0.023608,0.191048


In [17]:
df.isnull().sum()

Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

In [18]:
df.duplicated().sum()

0

In [21]:
X = df.drop(columns=["Diabetes_012"])
y = df["Diabetes_012"]

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [25]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts(normalize=True))

print("\nAfter SMOTE:")
print(y_train_smote.value_counts(normalize=True))

Before SMOTE:
Diabetes_012
0.0    0.827112
2.0    0.152744
1.0    0.020144
Name: proportion, dtype: float64

After SMOTE:
Diabetes_012
0.0    0.333333
2.0    0.333333
1.0    0.333333
Name: proportion, dtype: float64


In [27]:
# SMOTE was applied only to the training set to reduce class imbalance.
# The test set was kept unchanged for fair evaluation.

# Baseline model

## Logistics Regression

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Step 1: scale features
scaler = StandardScaler()
X_train_smote_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

# Step 2: define model
lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial'
)

# Step 3: train model
lr.fit(X_train_smote_scaled, y_train_smote)

# Step 4: predict on original test set
y_pred_lr = lr.predict(X_test_scaled)

# Step 5: evaluate
print("=== Logistic Regression Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Macro-F1:", f1_score(y_test, y_pred_lr, average='macro'))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

/Users/becki/.conda/envs/compsci760/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


=== Logistic Regression Results ===
Accuracy: 0.6254107100115325
Macro-F1: 0.4230899079992955

Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      0.64      0.76     38012
         1.0       0.03      0.31      0.06       926
         2.0       0.36      0.58      0.45      7019

    accuracy                           0.63     45957
   macro avg       0.45      0.51      0.42     45957
weighted avg       0.84      0.63      0.70     45957


Confusion Matrix:
[[24369  6853  6790]
 [  265   289   372]
 [ 1243  1692  4084]]


In [30]:
lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred_lr),
    "Macro_F1": f1_score(y_test, y_pred_lr, average='macro')
}

print(lr_results)

{'Model': 'Logistic Regression', 'Accuracy': 0.6254107100115325, 'Macro_F1': 0.4230899079992955}


## Random Forest

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Step 1: define model
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# Step 2: train model
rf.fit(X_train_smote, y_train_smote)

# Step 3: predict on original test set
y_pred_rf = rf.predict(X_test)

# Step 4: evaluate
print("=== Random Forest Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Macro-F1:", f1_score(y_test, y_pred_rf, average='macro'))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

=== Random Forest Results ===
Accuracy: 0.7523989816567661
Macro-F1: 0.4451947191287882

Classification Report:
              precision    recall  f1-score   support

         0.0       0.91      0.80      0.85     38012
         1.0       0.04      0.04      0.04       926
         2.0       0.36      0.59      0.45      7019

    accuracy                           0.75     45957
   macro avg       0.43      0.48      0.45     45957
weighted avg       0.81      0.75      0.77     45957


Confusion Matrix:
[[30373   729  6910]
 [  507    33   386]
 [ 2669   178  4172]]


In [32]:
rf_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Macro_F1": f1_score(y_test, y_pred_rf, average='macro')
}

print(rf_results)

{'Model': 'Random Forest', 'Accuracy': 0.7523989816567661, 'Macro_F1': 0.4451947191287882}


In [33]:
import pandas as pd

comparison_df = pd.DataFrame([lr_results, rf_results])
comparison_df

,Model,Accuracy,Macro_F1
0,Logistic Regression,0.625411,0.423090
1,Random Forest,0.752399,0.445195


## XGBoost

In [35]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Step 1: define model
xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    n_jobs=-1,
    random_state=42
)

# Step 2: train model
xgb.fit(X_train_smote, y_train_smote)

# Step 3: predict on original test set
y_pred_xgb = xgb.predict(X_test)

# Step 4: evaluate
print("=== XGBoost Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Macro-F1:", f1_score(y_test, y_pred_xgb, average='macro'))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

=== XGBoost Results ===
Accuracy: 0.8088648084078596
Macro-F1: 0.44603913494319175

Classification Report:
              precision    recall  f1-score   support

         0.0       0.89      0.89      0.89     38012
         1.0       0.00      0.00      0.00       926
         2.0       0.43      0.47      0.45      7019

    accuracy                           0.81     45957
   macro avg       0.44      0.45      0.45     45957
weighted avg       0.80      0.81      0.80     45957


Confusion Matrix:
[[33851     0  4161]
 [  659     0   267]
 [ 3697     0  3322]]


/Users/becki/.conda/envs/compsci760/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/becki/.conda/envs/compsci760/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/becki/.conda/envs/compsci760/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

In [36]:
xgb_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred_xgb),
    "Macro_F1": f1_score(y_test, y_pred_xgb, average='macro')
}

print(xgb_results)

{'Model': 'XGBoost', 'Accuracy': 0.8088648084078596, 'Macro_F1': 0.44603913494319175}


### comparison table

In [37]:
comparison_df = pd.DataFrame([lr_results, rf_results, xgb_results])
comparison_df

,Model,Accuracy,Macro_F1
0,Logistic Regression,0.625411,0.423090
1,Random Forest,0.752399,0.445195
2,XGBoost,0.808865,0.446039


# Subgroup fairness evaluation

In [38]:
sensitive_features = ["Sex", "Age", "Income"]
sensitive_test = X_test[sensitive_features].copy()

In [39]:
fairness_df = sensitive_test.copy()
fairness_df["y_true"] = y_test.values
fairness_df["lr_pred"] = y_pred_lr
fairness_df["rf_pred"] = y_pred_rf
fairness_df["xgb_pred"] = y_pred_xgb

fairness_df.head()

,Sex,Age,Income,y_true,lr_pred,rf_pred,xgb_pred
251778,1.0,9.0,1.0,2.0,2.0,2.0,2
147101,1.0,8.0,6.0,0.0,0.0,0.0,0
151531,1.0,9.0,7.0,0.0,2.0,0.0,0
129092,0.0,7.0,3.0,0.0,1.0,2.0,0
192671,0.0,9.0,7.0,2.0,1.0,0.0,0


### subgroup evaluation

In [40]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_by_group(df, group_col, pred_col):
    results = []
    
    for group_value in sorted(df[group_col].unique()):
        subset = df[df[group_col] == group_value]
        
        acc = accuracy_score(subset["y_true"], subset[pred_col])
        macro_f1 = f1_score(subset["y_true"], subset[pred_col], average="macro")
        
        results.append({
            "GroupFeature": group_col,
            "GroupValue": group_value,
            "Model": pred_col,
            "N": len(subset),
            "Accuracy": acc,
            "Macro_F1": macro_f1
        })
    
    return pd.DataFrame(results)

In [41]:
lr_sex = evaluate_by_group(fairness_df, "Sex", "lr_pred")
lr_age = evaluate_by_group(fairness_df, "Age", "lr_pred")
lr_income = evaluate_by_group(fairness_df, "Income", "lr_pred")

pd.concat([lr_sex, lr_age, lr_income], ignore_index=True)

,GroupFeature,GroupValue,Model,N,Accuracy,Macro_F1
0,Sex,0.0,lr_pred,25657,0.635616,0.426824
1,Sex,1.0,lr_pred,20300,0.612512,0.416880
2,Age,1.0,lr_pred,1071,0.976657,0.413031
3,Age,2.0,lr_pred,1432,0.939944,0.383926
4,Age,3.0,lr_pred,2036,0.905697,0.400783
5,Age,4.0,lr_pred,2473,0.862111,0.429402
6,Age,5.0,lr_pred,2854,0.826559,0.429786
7,Age,6.0,lr_pred,3417,0.761487,0.442951
8,Age,7.0,lr_pred,4600,0.696957,0.433174
9,Age,8.0,lr_pred,5476,0.635866,0.425631


In [42]:
rf_sex = evaluate_by_group(fairness_df, "Sex", "rf_pred")
rf_age = evaluate_by_group(fairness_df, "Age", "rf_pred")
rf_income = evaluate_by_group(fairness_df, "Income", "rf_pred")

pd.concat([rf_sex, rf_age, rf_income], ignore_index=True)

,GroupFeature,GroupValue,Model,N,Accuracy,Macro_F1
0,Sex,0.0,rf_pred,25657,0.765600,0.448428
1,Sex,1.0,rf_pred,20300,0.735714,0.440938
2,Age,1.0,rf_pred,1071,0.978525,0.381146
3,Age,2.0,rf_pred,1432,0.961592,0.374407
4,Age,3.0,rf_pred,2036,0.938605,0.390910
5,Age,4.0,rf_pred,2473,0.909017,0.411881
6,Age,5.0,rf_pred,2854,0.887176,0.421186
7,Age,6.0,rf_pred,3417,0.847820,0.433319
8,Age,7.0,rf_pred,4600,0.799565,0.440324
9,Age,8.0,rf_pred,5476,0.754931,0.445223


In [43]:
xgb_sex = evaluate_by_group(fairness_df, "Sex", "xgb_pred")
xgb_age = evaluate_by_group(fairness_df, "Age", "xgb_pred")
xgb_income = evaluate_by_group(fairness_df, "Income", "xgb_pred")

pd.concat([xgb_sex, xgb_age, xgb_income], ignore_index=True)

,GroupFeature,GroupValue,Model,N,Accuracy,Macro_F1
0,Sex,0.0,xgb_pred,25657,0.825311,0.449379
1,Sex,1.0,xgb_pred,20300,0.788079,0.441485
2,Age,1.0,xgb_pred,1071,0.985994,0.375425
3,Age,2.0,xgb_pred,1432,0.976257,0.372334
4,Age,3.0,xgb_pred,2036,0.956287,0.366109
5,Age,4.0,xgb_pred,2473,0.934088,0.406646
6,Age,5.0,xgb_pred,2854,0.915907,0.415538
7,Age,6.0,xgb_pred,3417,0.868891,0.418115
8,Age,7.0,xgb_pred,4600,0.840000,0.442791
9,Age,8.0,xgb_pred,5476,0.805332,0.442875


In [44]:
fairness_results = pd.concat(
    [lr_sex, lr_age, lr_income,
     rf_sex, rf_age, rf_income,
     xgb_sex, xgb_age, xgb_income],
    ignore_index=True
)

fairness_results = fairness_results.round(3)
fairness_results

,GroupFeature,GroupValue,Model,N,Accuracy,Macro_F1
0,Sex,0.0,lr_pred,25657,0.636,0.427
1,Sex,1.0,lr_pred,20300,0.613,0.417
2,Age,1.0,lr_pred,1071,0.977,0.413
3,Age,2.0,lr_pred,1432,0.940,0.384
4,Age,3.0,lr_pred,2036,0.906,0.401
...,...,...,...,...,...,...
64,Income,4.0,xgb_pred,3983,0.759,0.441
65,Income,5.0,xgb_pred,5096,0.782,0.445
66,Income,6.0,xgb_pred,6971,0.797,0.437
67,Income,7.0,xgb_pred,8055,0.821,0.433


In [45]:
gap_summary = fairness_results.groupby(["GroupFeature", "Model"]).agg(
    max_macro_f1=("Macro_F1", "max"),
    min_macro_f1=("Macro_F1", "min")
).reset_index()

gap_summary["gap"] = gap_summary["max_macro_f1"] - gap_summary["min_macro_f1"]
gap_summary = gap_summary.round(3)
gap_summary

,GroupFeature,Model,max_macro_f1,min_macro_f1,gap
0,Age,lr_pred,0.443,0.275,0.168
1,Age,rf_pred,0.448,0.374,0.074
2,Age,xgb_pred,0.452,0.366,0.086
3,Income,lr_pred,0.429,0.377,0.052
4,Income,rf_pred,0.452,0.424,0.028
5,Income,xgb_pred,0.453,0.433,0.020
6,Sex,lr_pred,0.427,0.417,0.010
7,Sex,rf_pred,0.448,0.441,0.007
8,Sex,xgb_pred,0.449,0.441,0.008


In [46]:
gap_summary["FairnessSummary"] = gap_summary["gap"].apply(
    lambda x: "Small gap" if x < 0.03 else ("Moderate gap" if x < 0.08 else "Large gap")
)

gap_summary

,GroupFeature,Model,max_macro_f1,min_macro_f1,gap,FairnessSummary
0,Age,lr_pred,0.443,0.275,0.168,Large gap
1,Age,rf_pred,0.448,0.374,0.074,Moderate gap
2,Age,xgb_pred,0.452,0.366,0.086,Large gap
3,Income,lr_pred,0.429,0.377,0.052,Moderate gap
4,Income,rf_pred,0.452,0.424,0.028,Small gap
5,Income,xgb_pred,0.453,0.433,0.020,Small gap
6,Sex,lr_pred,0.427,0.417,0.010,Small gap
7,Sex,rf_pred,0.448,0.441,0.007,Small gap
8,Sex,xgb_pred,0.449,0.441,0.008,Small gap


The subgroup fairness analysis shows that model performance is relatively consistent across sex groups, but less consistent across age and income groups. Among the three sensitive features, age has the largest Macro-F1 gap across all models, indicating the strongest subgroup disparity. Logistic Regression is most sensitive to age-related differences, while Random Forest provides a more balanced trade-off between predictive performance and subgroup consistency. Although XGBoost achieves the strongest overall performance, it does not eliminate fairness disparities, especially across age groups.

# Subgroup Fairness Evaluation

In [48]:
# clean model names
model_name_map = {
    "lr_pred": "Logistic Regression",
    "rf_pred": "Random Forest",
    "xgb_pred": "XGBoost"
}

gap_summary_clean = gap_summary.copy()
gap_summary_clean["Model"] = gap_summary_clean["Model"].map(model_name_map)

gap_summary_clean = gap_summary_clean.rename(columns={
    "GroupFeature": "SensitiveFeature",
    "max_macro_f1": "BestGroup_MacroF1",
    "min_macro_f1": "WorstGroup_MacroF1",
    "gap": "MacroF1_Gap"
})

gap_summary_clean = gap_summary_clean.round(3)
gap_summary_clean

,SensitiveFeature,Model,BestGroup_MacroF1,WorstGroup_MacroF1,MacroF1_Gap,FairnessSummary
0,Age,Logistic Regression,0.443,0.275,0.168,Large gap
1,Age,Random Forest,0.448,0.374,0.074,Moderate gap
2,Age,XGBoost,0.452,0.366,0.086,Large gap
3,Income,Logistic Regression,0.429,0.377,0.052,Moderate gap
4,Income,Random Forest,0.452,0.424,0.028,Small gap
5,Income,XGBoost,0.453,0.433,0.020,Small gap
6,Sex,Logistic Regression,0.427,0.417,0.010,Small gap
7,Sex,Random Forest,0.448,0.441,0.007,Small gap
8,Sex,XGBoost,0.449,0.441,0.008,Small gap


In [49]:
fairness_pivot = gap_summary_clean.pivot(
    index="SensitiveFeature",
    columns="Model",
    values="MacroF1_Gap"
).round(3)

fairness_pivot

Model,Logistic Regression,Random Forest,XGBoost
SensitiveFeature,,,
Age,0.168,0.074,0.086
Income,0.052,0.028,0.020
Sex,0.010,0.007,0.008


In [50]:
comparison_df.round(3).to_csv("model_comparison_results.csv", index=False)
gap_summary_clean.to_csv("fairness_gap_summary.csv", index=False)
fairness_pivot.to_csv("fairness_gap_pivot.csv")

# Explainability

In [53]:
import numpy as np
import pandas as pd

shap_array = np.array(shap_values_rf)
print("SHAP array shape:", shap_array.shape)

# Case 1: binary / multiclass returned as list-like converted to 3D
if shap_array.ndim == 3:
    # average over samples and classes
    shap_abs_mean = np.abs(shap_array).mean(axis=(0, 2))
    
# Case 2: returned as 2D (samples, features)
elif shap_array.ndim == 2:
    shap_abs_mean = np.abs(shap_array).mean(axis=0)

# Case 3: other format
else:
    raise ValueError(f"Unexpected SHAP shape: {shap_array.shape}")

rf_shap_importance = pd.DataFrame({
    "Feature": X_shap.columns,
    "MeanAbsSHAP": shap_abs_mean
}).sort_values("MeanAbsSHAP", ascending=False)

rf_shap_importance.head(10)

SHAP array shape: (1000, 21, 3)


,Feature,MeanAbsSHAP
0,HighBP,0.073459
1,HighChol,0.051692
13,GenHlth,0.041044
18,Age,0.019192
3,BMI,0.015341
16,DiffWalk,0.013968
17,Sex,0.007378
7,PhysActivity,0.006539
6,HeartDiseaseorAttack,0.006096
20,Income,0.004951


In the Random Forest model, the most influential features were HighBP, HighChol, and GenHlth, followed by Age and BMI. This suggests that the model relies strongly on clinically plausible health indicators rather than arbitrary variables.

Age is not only an important predictive feature, but also the subgroup dimension associated with the largest performance disparity.

Age appears in both analyses: it is one of the most important predictive features in SHAP, and it is also the subgroup attribute with the largest Macro-F1 gap. This suggests that age plays a central role in both model prediction and subgroup disparity.

Sex and Income also appeared in the top 10 features, although their average contribution was smaller than major health-related variables.